# IS310 Group 6 — State Immigration Law Sentiment Analysis
**Daria Meshcheriakova | UIUC Spring 2026 | Culture as Data**

---

## Research Question
**Which state — Illinois, California, or Arizona — is most pro-immigration, and how has each state's legislative stance shifted over time (2009–2026)?**

## Workflow Summary
1. **Data Collection** — 589 total laws collected from NCSL (64), LegiScan API (337), Arizona Legislature API (183), and state portals via Plural Policy research (5)
2. **Manual Labeling** — 30 laws labeled Pro-Immigration / Anti-Immigration; 21 were read from full bill text with written reasoning for each; 9 labeled from law descriptions
3. **TF-IDF Feature Extraction** — each law's description converted to numerical word-frequency features
4. **Logistic Regression** — model trained on 30 labeled laws; Leave-One-Out CV measures accuracy
5. **Scoring All Laws** — model scores all 589 laws on a 0–1 pro-immigration probability scale
6. **Trend Analysis** — scores analyzed by state over time to answer the research question

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

PROJECT = '/Users/dariam/Desktop/IS310 - Intial Dataset Submission'
STATE_COLORS  = {'Arizona': '#e74c3c', 'California': '#2980b9', 'Illinois': '#27ae60'}
STATE_MARKERS = {'Arizona': 'o', 'California': 's', 'Illinois': '^'}

In [ ]:
df = pd.read_csv(f'{PROJECT}/final_dataset_3.csv', dtype=str)

# Identify data source: LegiScan vs NCSL
if 'source' in df.columns:
    df['data_source'] = df['source'].fillna('').apply(
        lambda x: 'LegiScan' if 'LegiScan' in str(x) else 'NCSL'
    )
elif 'bill_id' in df.columns:
    df['data_source'] = df['bill_id'].fillna('').apply(
        lambda x: 'LegiScan' if str(x).strip() not in ('', 'nan') else 'NCSL'
    )
else:
    df['data_source'] = 'Unknown'

# Numeric conversions
df['Year'] = pd.to_numeric(df['Year'], errors='coerce')
df['pro_immigration_score'] = pd.to_numeric(df['pro_immigration_score'], errors='coerce')

print(f'Total laws in dataset: {len(df)}')
print()
print('Laws by state:')
print(df['State'].value_counts().to_string())
print()
print('Data source breakdown:')
print(df['data_source'].value_counts().to_string())
print()
print('Manual labels (my_label column):')
print(df['my_label'].value_counts(dropna=False).to_string())
print()
desc_filled = df['Law Description'].notna().sum()
print(f'Law Description filled: {desc_filled} / {len(df)} rows')
df.head(3)

## Section 1 — Data Collection: How the Dataset Was Built

The dataset combines four sources:
- **NCSL (National Conference of State Legislatures)** — 64 laws collected manually from the NCSL Immigration Legislation Archived Database
- **LegiScan API** — 337 laws fetched programmatically via the LegiScan free-tier API across multiple immigration-related keyword searches
- **Arizona Legislature API** — 183 laws retrieved via the apps.azleg.gov JSON API (bill titles and dispositions only — no full descriptions)
- **State portals via Plural Policy research** — 5 laws added manually after cross-checking with the Plural Policy / Open States interface

**Total: 589 laws | Arizona: 238 | California: 179 | Illinois: 172 | Years: 2009–2026**

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Chart 1: NCSL vs LegiScan by state
source_state = df.groupby(['State', 'data_source']).size().unstack(fill_value=0)
for col in ['NCSL', 'LegiScan']:
    if col not in source_state.columns:
        source_state[col] = 0
source_state[['NCSL', 'LegiScan']].plot(
    kind='bar', ax=axes[0],
    color=['#95a5a6', '#3498db'], rot=0, edgecolor='white'
)
axes[0].set_title('Laws by Source & State', fontsize=12)
axes[0].set_xlabel('')
axes[0].set_ylabel('Number of Laws')
axes[0].legend(title='Source')
for container in axes[0].containers:
    axes[0].bar_label(container, padding=2)

# Chart 2: Manually labeled vs model-predicted
label_counts = df.groupby('State').apply(
    lambda g: pd.Series({
        'Manually Labeled': g['my_label'].isin(['Pro-Immigration', 'Anti-Immigration']).sum(),
        'Model-Predicted':  (~g['my_label'].isin(['Pro-Immigration', 'Anti-Immigration'])).sum()
    })
)
label_counts.plot(
    kind='bar', ax=axes[1],
    color=['#2980b9', '#bdc3c7'], rot=0, edgecolor='white'
)
axes[1].set_title('Labeled vs Model-Predicted by State', fontsize=12)
axes[1].set_xlabel('')
axes[1].set_ylabel('Number of Laws')
axes[1].legend()
for container in axes[1].containers:
    axes[1].bar_label(container, padding=2)

# Chart 3: Laws per year
yearly = df.groupby('Year').size()
axes[2].bar(yearly.index, yearly.values, color='#7f8c8d', edgecolor='white')
axes[2].set_title('Laws in Dataset by Year (2009–2026)', fontsize=12)
axes[2].set_xlabel('Year')
axes[2].set_ylabel('Number of Laws')
axes[2].set_xlim(2008, 2027)

plt.tight_layout()
plt.savefig(f'{PROJECT}/fig1_data_overview.png', bbox_inches='tight')
plt.show()
print('Saved: fig1_data_overview.png')

## Section 2 — Manual Labeling

**30 laws** were manually labeled (21 Pro-Immigration, 9 Anti-Immigration):

- **Pro-Immigration** — the law expands rights, protections, or access for immigrants
- **Anti-Immigration** — the law restricts rights, increases enforcement, or creates barriers

Of these 30, **21 were labeled by reading the full bill text** and recording a written justification for each decision. The remaining 9 were labeled by reading the law description. All 30 labeled laws are used together to train the classifier in Section 3.

### Labeling Rubric — What Counts as Pro vs. Anti Immigration?

The labeling decision was based on **how a law treats immigrants and immigration as a whole** — not on whether the law specifically targets documented or undocumented immigrants. That distinction was intentional. Many laws do not specify legal status at all, and what mattered was the overall framing, language, and effect of the law regardless of which type of immigrant it referenced.

**A law was labeled Pro-Immigration if it did one or more of the following:**
- Uses **supportive or welcoming language** toward immigrants as a group
- **Protects immigrants** from enforcement actions (e.g., limiting ICE cooperation, protecting sensitive locations such as schools, churches, and hospitals from enforcement)
- **Extends rights, benefits, or access** — including drivers licenses, healthcare, education, legal aid, or public benefits
- **Allocates funding or resources** to support immigrant communities or integration
- Creates **legal recourse** for immigrants who face discrimination or enforcement abuse

**A law was labeled Anti-Immigration if it did one or more of the following:**
- **Expands enforcement** against immigrants or increases penalties tied to immigration status
- **Restricts access** to employment, housing, benefits, or services based on immigration status
- **Deepens state cooperation** with federal immigration enforcement (e.g., data sharing with ICE, 287(g) agreements)
- **Criminalizes** behaviors such as harboring, transporting, or employing undocumented people
- Uses **punitive language** that frames immigration primarily as a legal violation or public safety threat

**Borderline cases** were decided by intent and effect rather than surface language. A law written to appear neutral but designed to preserve an enforcement regime was labeled Anti-Immigration. A law that mentions immigration status in the context of expanding protections was labeled Pro-Immigration.

---

> **Example of a genuinely ambiguous case — AZ SB 1474 (2026):**
> *"Immigration laws; local enforcement; training."*
> This bill was difficult to label from the title alone. The phrase could mean training local agencies to *protect and comply with* immigration law (pro-immigrant framing) or training them to *enforce immigration law against* immigrants (anti-immigrant framing). After reviewing the available context — specifically that "local enforcement; training" framing aligns with building enforcement capacity — it was labeled **Anti-Immigration** because training local agencies to implement immigration enforcement supports a restrictive rather than protective posture. This case illustrates why short titles from the Arizona Legislature API are unreliable inputs for the TF-IDF model (the model predicted it Pro-Immigration in LOO CV, which is one of the 6 misclassifications) and why those rows are treated as uncertain in the analysis.

In [ ]:
labeled = df[df['my_label'].isin(['Pro-Immigration', 'Anti-Immigration'])].copy()

print(f'Manually labeled laws: {len(labeled)}')
print()
print('Label breakdown overall:')
print(labeled['my_label'].value_counts().to_string())
print()
print('Breakdown by state:')
print(labeled.groupby(['State', 'my_label']).size().unstack(fill_value=0).to_string())

## Section 3 — TF-IDF + Logistic Regression Classifier

**How it works:**
1. **TF-IDF** converts each law's description into word/phrase importance scores — words that appear often in one law but rarely across all laws score higher
2. **Logistic Regression** learns which words predict Pro-Immigration vs. Anti-Immigration using the 30 labeled laws (21 read from full bill text + 9 labeled from descriptions)
3. **Leave-One-Out CV (LOO CV)** tests fairly on a small dataset: train on all-but-one laws, predict the held-out one, repeat for every labeled law

**Settings:** bigrams (1–2 word phrases), `class_weight='balanced'` (handles class imbalance between Pro and Anti), 500 max features

In [ ]:
labeled_clean = labeled.dropna(subset=['Law Description']).copy()
X = labeled_clean['Law Description']
y = labeled_clean['my_label']

print(f'Training rows (has description + manual label): {len(labeled_clean)}')

clf = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=300,
        stop_words='english',
        ngram_range=(1, 2),
        sublinear_tf=True
    )),
    ('lr', LogisticRegression(
        max_iter=1000,
        C=1.0,
        class_weight='balanced',
        solver='lbfgs'
    ))
])

# Leave-One-Out cross-validation
loo = LeaveOneOut()
y_pred_loo   = cross_val_predict(clf, X, y, cv=loo)
y_proba_loo  = cross_val_predict(clf, X, y, cv=loo, method='predict_proba')

acc = accuracy_score(y, y_pred_loo)
print(f'\nLeave-One-Out CV Accuracy: {acc:.1%}  ({(y == y_pred_loo).sum()} / {len(y)} correct)')
print()
print(classification_report(y, y_pred_loo))

# Fit final model on all labeled data (used to score unlabeled laws)
clf.fit(X, y)
class_order = list(clf.classes_)
pro_idx = class_order.index('Pro-Immigration')
print(f'Classes: {class_order}  |  Pro-Immigration index: {pro_idx}')

In [ ]:
cm = confusion_matrix(y, y_pred_loo, labels=['Anti-Immigration', 'Pro-Immigration'])

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(['Anti', 'Pro'], fontsize=11)
ax.set_yticklabels(['Anti', 'Pro'], fontsize=11)
ax.set_xlabel('Predicted', fontsize=11)
ax.set_ylabel('True (Researcher Label)', fontsize=11)
ax.set_title('Confusion Matrix — LOO Cross-Validation', fontsize=12)
for i in range(2):
    for j in range(2):
        color = 'white' if cm[i, j] > cm.max() / 2 else 'black'
        ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                fontsize=16, fontweight='bold', color=color)
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig(f'{PROJECT}/fig2_confusion_matrix.png', bbox_inches='tight')
plt.show()
print('Saved: fig2_confusion_matrix.png')

### Feature Importance — What Words Drive Each Classification?

The bars show the logistic regression coefficients for the top TF-IDF features. Positive coefficients push toward Pro-Immigration; negative push toward Anti-Immigration.

In [ ]:
feat_names = np.array(clf.named_steps['tfidf'].get_feature_names_out())
coef = clf.named_steps['lr'].coef_[0]
top_n = 12

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

top_pro_idx  = np.argsort(coef)[-top_n:]
axes[0].barh(feat_names[top_pro_idx], coef[top_pro_idx], color='#2980b9', edgecolor='white')
axes[0].set_title('Top Words → Pro-Immigration', fontsize=12)
axes[0].set_xlabel('Logistic Regression Coefficient')
axes[0].axvline(0, color='gray', linewidth=0.8)

top_anti_idx = np.argsort(coef)[:top_n]
axes[1].barh(feat_names[top_anti_idx], coef[top_anti_idx], color='#e74c3c', edgecolor='white')
axes[1].set_title('Top Words → Anti-Immigration', fontsize=12)
axes[1].set_xlabel('Logistic Regression Coefficient')
axes[1].axvline(0, color='gray', linewidth=0.8)

plt.suptitle('TF-IDF Feature Importance (trained on 66 manually labeled laws)',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(f'{PROJECT}/fig3_feature_importance.png', bbox_inches='tight')
plt.show()
print('Saved: fig3_feature_importance.png')

## Section 4 — Scoring All 589 Laws

Every law with a description gets a **pro-immigration probability score** (0 = strongly anti-immigration, 1 = strongly pro-immigration):

- **Labeled laws** → scored using **LOO out-of-fold probabilities** (each law scored by a model that never saw it during training — unbiased estimate). Of these, **21 were manually labeled by reading full bill text with written reasoning**; the other 9 were labeled from law descriptions.
- **Unlabeled laws** → scored using the **full model** trained on all 30 labeled laws

All 589 laws have descriptions and receive a score.

In [ ]:
# --- Labeled rows: use LOO out-of-fold scores ---
scored_labeled = labeled_clean.copy()
scored_labeled['pro_score']      = y_proba_loo[:, pro_idx]
scored_labeled['effective_label'] = y_pred_loo
scored_labeled['label_source']   = 'Manual'

# --- Unlabeled rows that have a description: use full model ---
unlabeled = df[~df['my_label'].isin(['Pro-Immigration', 'Anti-Immigration'])].copy()
unlabeled_desc = unlabeled.dropna(subset=['Law Description']).copy()

if len(unlabeled_desc) > 0:
    proba = clf.predict_proba(unlabeled_desc['Law Description'])
    unlabeled_desc['pro_score']      = proba[:, pro_idx]
    unlabeled_desc['effective_label'] = clf.predict(unlabeled_desc['Law Description'])
    unlabeled_desc['label_source']   = 'Model-Predicted'

# --- Combine ---
all_scored = pd.concat([scored_labeled, unlabeled_desc], ignore_index=True)

print(f'Total scored: {len(all_scored)} / {len(df)} laws')
print()
print('Average pro-immigration score by state:')
summary = all_scored.groupby('State')['pro_score'].agg(['mean', 'count']).round(3)
summary.columns = ['avg_score', 'n_scored']
print(summary.to_string())
print()
print('Effective label distribution (manual + model-predicted):')
print(all_scored['effective_label'].value_counts().to_string())

In [ ]:
display_cols = ['State', 'Year', 'Law Passed', 'my_label', 'effective_label', 'pro_score', 'label_source']
all_scored[display_cols].sort_values(['State', 'Year']).reset_index(drop=True)

## Section 5 — Trend Analysis (2009–2026)

Each dot is one law. Lines connect yearly averages. The dashed line at 0.5 is the neutral threshold.

**Effective label used**: manual label where available; model-predicted label for unlabeled laws.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6))

for state, grp in all_scored.groupby('State'):
    color  = STATE_COLORS.get(state, '#999')
    marker = STATE_MARKERS.get(state, 'o')

    # Individual law dots
    ax.scatter(grp['Year'], grp['pro_score'],
               color=color, marker=marker, alpha=0.4, s=65, zorder=3)

    # Yearly average line
    yearly_mean = grp.groupby('Year')['pro_score'].mean()
    ax.plot(yearly_mean.index, yearly_mean.values,
            color=color, marker=marker, markersize=9,
            linewidth=2.2, label=state, zorder=4)

ax.axhline(0.5, color='gray', linestyle='--', linewidth=1.2, alpha=0.7, label='Neutral (0.5)')
ax.fill_between([2008, 2027], 0.5, 1.0, alpha=0.04, color='#2980b9')
ax.fill_between([2008, 2027], 0.0, 0.5, alpha=0.04, color='#e74c3c')
ax.text(2008.3, 0.97, 'Pro-Immigration zone', color='#2980b9', fontsize=9, va='top')
ax.text(2008.3, 0.03, 'Anti-Immigration zone', color='#e74c3c', fontsize=9, va='bottom')

ax.set_xlim(2008, 2027)
ax.set_ylim(0, 1.0)
ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Pro-Immigration Probability Score', fontsize=12)
ax.set_title(
    'Immigration Law Sentiment by State (2009–2026)\n'
    'Dots = individual laws | Lines = yearly average | Scored by TF-IDF + Logistic Regression',
    fontsize=13
)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.25)

plt.tight_layout()
plt.savefig(f'{PROJECT}/fig4_trend_by_state.png', bbox_inches='tight')
plt.show()
print('Saved: fig4_trend_by_state.png')

## Section 6 — Overall State Comparison

In [ ]:
state_avg = all_scored.groupby('State')['pro_score'].mean().sort_values()

fig, ax = plt.subplots(figsize=(7, 4))
bar_colors = [STATE_COLORS.get(s, '#999') for s in state_avg.index]
bars = ax.barh(state_avg.index, state_avg.values,
               color=bar_colors, edgecolor='white', height=0.5)
ax.axvline(0.5, color='gray', linestyle='--', linewidth=1.2, alpha=0.8, label='Neutral (0.5)')
ax.set_xlabel('Average Pro-Immigration Score (2009–2026)', fontsize=11)
ax.set_title('Which State is Most Pro-Immigration?', fontsize=13)
ax.set_xlim(0, 1)
for bar, val in zip(bars, state_avg.values):
    ax.text(val + 0.013, bar.get_y() + bar.get_height() / 2,
            f'{val:.3f}', va='center', fontsize=11, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig(f'{PROJECT}/fig5_state_comparison.png', bbox_inches='tight')
plt.show()
print('Saved: fig5_state_comparison.png')

## Detailed Per-State Score Tables

Full sorted lists for each state, showing manual labels vs. model-predicted labels and each law's pro-immigration score.

In [ ]:
for state in ['California', 'Illinois', 'Arizona']:
    sub = (all_scored[all_scored['State'] == state]
           [['Year', 'Law Passed', 'my_label', 'effective_label', 'pro_score', 'label_source']]
           .sort_values('Year')
           .reset_index(drop=True)
           .copy())
    sub['pro_score']  = sub['pro_score'].round(3)
    sub['Law Passed'] = sub['Law Passed'].str[:55]

    n_manual = (sub['label_source'] == 'Manual').sum()
    n_pro    = (sub['effective_label'] == 'Pro-Immigration').sum()
    n_anti   = (sub['effective_label'] == 'Anti-Immigration').sum()

    print(f'\n{"="*68}')
    print(f'  {state}  (n={len(sub)} scored | {n_manual} manually labeled | Pro: {n_pro} | Anti: {n_anti})')
    print(f'  Avg pro score: {sub["pro_score"].mean():.3f}')
    print('='*68)
    print(sub.to_string(index=False))

## Section 7 — Conclusions

### Research Question
**Which state — Illinois, California, or Arizona — is most pro-immigration, and how has each state's legislative stance shifted over time (2009–2026)?**

### Answer

| State | Avg Score | Assessment |
|-------|-----------|------------|
| Illinois | ~0.535 | Highest pro-immigration |
| California | ~0.527 | Strongly pro-immigration |
| Arizona | ~0.508 | Unreliable* — see note |

**Illinois** and **California** are both strongly pro-immigration throughout the study period based on the laws that had sufficient text for the model to analyze. **Arizona's** overall model average is not a reliable indicator because 183 of its 238 laws came from the Arizona Legislature API and have only 3–7 word titles — not enough text for TF-IDF to score meaningfully. Among the 13 Arizona laws that were manually labeled, 9 are Anti-Immigration and 4 are Pro-Immigration, which better reflects Arizona's actual legislative posture.

### Model Performance
The TF-IDF + Logistic Regression classifier achieved **80.0% leave-one-out accuracy** (24/30 correct) on the labeled laws. The 6 misclassified laws are the borderline cases — laws where coded language, procedural framing, or dual-purpose provisions made the stance difficult even for the researcher to determine. This shows the model struggles precisely where human judgment also struggles.

### Manual Labeling
**30 laws** were manually labeled (21 Pro-Immigration, 9 Anti-Immigration). Of these, **21 were labeled by reading full bill text** with a written justification for each decision — these represent the deepest human review. The other 9 were labeled by reading the law description. All 30 are used together to train the classifier.

### Shifts Over Time
- **Arizona**: Heavily anti-immigration 2009–2011 (SB 1070 era); model scores for post-2011 Arizona laws are unreliable due to short-title text
- **California**: Consistently pro-immigration throughout; variation reflects older laws using procedural rather than explicitly protective language
- **Illinois**: All dataset laws are from 2019–2026, the post-Trump-era period with the most explicitly pro-immigrant statutory language in state history

### Dataset
- **589 total laws**: 64 from NCSL, 337 from LegiScan API, 183 from Arizona Legislature API, 5 from state portals via Plural Policy research
- **30 laws** manually labeled (21 read from full bill text with reasoning; 9 from descriptions)
- Arizona: 238 laws | California: 179 laws | Illinois: 172 laws
- All 589 laws have descriptions and receive a score

### Limitations
- Arizona Legislature API returns only 3–7 word bill titles, not descriptions — TF-IDF scores for those 183 rows cluster near 0.50 and should not be interpreted as findings
- Only 21 laws had full bill text review; others scored from 1–3 sentence descriptions, so nuance is lost
- Class imbalance (more Pro than Anti) limits anti-immigration recall despite balanced class weighting

### Data Sources & Citations

| Source | URL | Notes |
|--------|-----|-------|
| NCSL Immigration Legislation Archived Database | https://www.ncsl.org/immigration/immigration-legislation-database | Non-partisan state legislative tracker; primary manual collection source |
| LegiScan API | https://legiscan.com | CC BY 4.0; bulk bill metadata and descriptions for IL, CA, AZ |
| Arizona Legislature public API | https://apps.azleg.gov/api/ | Bill titles and dispositions by session; no full descriptions |
| Plural Policy / Open States | https://pluralpolicy.com | Legislative tracking interface; used to cross-check 2025–2026 bills |

### AI Assistance

All code in this notebook was written with the assistance of **Claude Code** (Anthropic, claude-sonnet-4-6), an AI coding assistant. Every prompt submitted during this project is logged in the Prompts codebook:

> `process/Prompts.md` — full record of every user prompt across all sessions, organized by date and goal.

The researcher made all labeling decisions, methodology choices, and interpretive calls. Claude Code was used as a coding tool: building scrapers, writing the TF-IDF pipeline, debugging, and formatting outputs.

---

## Section 8 — Visualizations

Four charts that together answer the core research question: *which state is most pro-immigration, and how has each state's stance shifted over time?*

- **Chart 1** — Line: average pro-immigration score per year, one line per state
- **Chart 2** — Horizontal bar: overall state ranking by average score
- **Chart 3** — Stacked bar: Pro vs Anti law counts and percentages per state
- **Chart 4** — Heatmap: score intensity across state × time period

In [ ]:
# ── Chart 1: Line — average pro-immigration score per year by state ───────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

COLORS  = {'California': '#2980b9', 'Illinois': '#27ae60', 'Arizona': '#e74c3c'}
STATES  = ['California', 'Illinois', 'Arizona']

fig, ax = plt.subplots(figsize=(12, 5))

for state in STATES:
    sub = all_scored[all_scored['State'] == state].copy()
    yearly = sub.groupby('Year')['pro_score'].mean().sort_index()
    ax.plot(yearly.index, yearly.values,
            marker='o', markersize=7, linewidth=2.5,
            color=COLORS[state], label=state, zorder=4)
    # Shade ±1 std
    yearly_std = sub.groupby('Year')['pro_score'].std().sort_index().fillna(0)
    ax.fill_between(yearly.index,
                    (yearly - yearly_std).clip(0, 1),
                    (yearly + yearly_std).clip(0, 1),
                    color=COLORS[state], alpha=0.12)

ax.axhline(0.5, color='#7f8c8d', linestyle='--', linewidth=1.3, alpha=0.8, label='Neutral (0.5)')
ax.set_ylim(0, 1)
ax.set_xlim(all_scored['Year'].min() - 0.5, all_scored['Year'].max() + 0.5)
ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Avg Pro-Immigration Score', fontsize=12)
ax.set_title('How Each State\'s Pro-Immigration Stance Changed Over Time (2009–2026)\n'
             'Shaded band = ±1 standard deviation across laws in that year',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.25)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f'))

plt.tight_layout()
plt.savefig(f'{PROJECT}/viz1_trend_line.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: viz1_trend_line.png')

In [ ]:
# ── Chart 2: Horizontal bar — which state is most pro-immigration? ────────────
import matplotlib.pyplot as plt
import numpy as np

COLORS = {'California': '#2980b9', 'Illinois': '#27ae60', 'Arizona': '#e74c3c'}

state_stats = (all_scored.groupby('State')['pro_score']
               .agg(['mean', 'sem', 'count'])
               .sort_values('mean'))

fig, ax = plt.subplots(figsize=(8, 4))

states = state_stats.index.tolist()
means  = state_stats['mean'].values
errors = state_stats['sem'].values * 1.96   # 95% confidence interval

bars = ax.barh(states, means,
               xerr=errors, capsize=6,
               color=[COLORS.get(s, '#999') for s in states],
               edgecolor='white', height=0.45,
               error_kw={'elinewidth': 1.8, 'ecolor': '#555', 'capthick': 1.8})

ax.axvline(0.5, color='#7f8c8d', linestyle='--', linewidth=1.4, label='Neutral (0.5)')

for bar, val, n in zip(bars, means, state_stats['count'].values):
    ax.text(val + 0.032, bar.get_y() + bar.get_height() / 2,
            f'{val:.3f}  (n={n})', va='center', fontsize=11, fontweight='bold')

ax.set_xlim(0, 1)
ax.set_xlabel('Average Pro-Immigration Score', fontsize=12)
ax.set_title('Which State Is Most Pro-Immigration?\n'
             'Average score across all scored laws · bars show 95% CI',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(axis='x', alpha=0.25)

plt.tight_layout()
plt.savefig(f'{PROJECT}/viz2_state_ranking.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: viz2_state_ranking.png')

In [ ]:
# ── Chart 3: Stacked bar — Pro vs Anti law counts per state ──────────────────
import matplotlib.pyplot as plt
import numpy as np

COLORS = {'California': '#2980b9', 'Illinois': '#27ae60', 'Arizona': '#e74c3c'}
STATES = ['California', 'Illinois', 'Arizona']

pro_counts  = [( all_scored[(all_scored['State'] == s) &
                             (all_scored['effective_label'] == 'Pro-Immigration')]).shape[0]
               for s in STATES]
anti_counts = [(all_scored[(all_scored['State'] == s) &
                            (all_scored['effective_label'] == 'Anti-Immigration')]).shape[0]
               for s in STATES]

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(STATES))

ax.bar(x, pro_counts,  color='#27ae60', alpha=0.88, label='Pro-Immigration',  width=0.5)
ax.bar(x, anti_counts, color='#e74c3c', alpha=0.88, label='Anti-Immigration', width=0.5,
       bottom=pro_counts)

for i, (p, a) in enumerate(zip(pro_counts, anti_counts)):
    total = p + a
    if p > 0:
        ax.text(i, p / 2,     f'{p}',              ha='center', va='center',
                color='white', fontsize=12, fontweight='bold')
    if a > 0:
        ax.text(i, p + a / 2, f'{a}',              ha='center', va='center',
                color='white', fontsize=12, fontweight='bold')
    ax.text(i, total + 3,     f'{p/total:.0%} pro', ha='center', va='bottom',
            fontsize=10, color='#2c3e50', fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(STATES, fontsize=12)
ax.set_ylabel('Number of Laws', fontsize=12)
ax.set_title('Pro vs Anti-Immigration Law Count by State\n'
             '(model-predicted labels for unlabeled laws)',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.25)

plt.tight_layout()
plt.savefig(f'{PROJECT}/viz3_pro_anti_stacked.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: viz3_pro_anti_stacked.png')

In [ ]:
# ── Chart 4: Heatmap — pro-immigration score by state × time period ───────────
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import pandas as pd

STATES = ['California', 'Illinois', 'Arizona']

# Group years into 3-year periods for a readable heatmap
def period(y):
    if   y <= 2011: return '2009–2011'
    elif y <= 2014: return '2012–2014'
    elif y <= 2017: return '2015–2017'
    elif y <= 2020: return '2018–2020'
    elif y <= 2023: return '2021–2023'
    else:           return '2024–2026'

all_scored['period'] = all_scored['Year'].apply(period)
PERIODS = ['2009–2011','2012–2014','2015–2017','2018–2020','2021–2023','2024–2026']

# Build matrix: rows=states, cols=periods
matrix = np.full((len(STATES), len(PERIODS)), np.nan)
counts = np.zeros((len(STATES), len(PERIODS)), dtype=int)

for i, state in enumerate(STATES):
    for j, period_label in enumerate(PERIODS):
        mask = (all_scored['State'] == state) & (all_scored['period'] == period_label)
        vals = all_scored.loc[mask, 'pro_score']
        if len(vals) > 0:
            matrix[i, j] = vals.mean()
            counts[i, j] = len(vals)

fig, ax = plt.subplots(figsize=(11, 4))

# Red→white→blue diverging colormap centered at 0.5
cmap = mcolors.LinearSegmentedColormap.from_list(
    'rw b', ['#c0392b', '#f5f5f5', '#2980b9'], N=256)

im = ax.imshow(matrix, cmap=cmap, vmin=0, vmax=1, aspect='auto')

ax.set_xticks(range(len(PERIODS)))
ax.set_xticklabels(PERIODS, fontsize=11)
ax.set_yticks(range(len(STATES)))
ax.set_yticklabels(STATES, fontsize=12)

# Annotate each cell with score and count
for i in range(len(STATES)):
    for j in range(len(PERIODS)):
        if not np.isnan(matrix[i, j]):
            val = matrix[i, j]
            text_color = 'white' if (val < 0.25 or val > 0.75) else '#2c3e50'
            ax.text(j, i, f'{val:.2f}\n(n={counts[i,j]})',
                    ha='center', va='center', fontsize=9.5,
                    color=text_color, fontweight='bold')
        else:
            ax.text(j, i, 'no data', ha='center', va='center',
                    fontsize=8, color='#aaa')

cbar = plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label('Avg Pro-Immigration Score', fontsize=10)
cbar.set_ticks([0, 0.25, 0.5, 0.75, 1.0])
cbar.ax.axhline(0.5, color='black', linewidth=1.2)

ax.set_title('Pro-Immigration Score Heatmap: State × Time Period\n'
             'Red = Anti-Immigration  ·  Blue = Pro-Immigration  ·  White = Neutral (0.5)',
             fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig(f'{PROJECT}/viz4_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: viz4_heatmap.png')